# Text Query

### RevRoad `Dataset` Locations:

- https://huggingface.co/RevolutionCrossroads
- https://huggingface.co/datasets/RevolutionCrossroads/loc_chronicling_america_1770-1810 (247 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/loc_chronicling_america_1770-1810_issues (707 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/nara_revolutionary_war_pension_files (9.4 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/nara_revolutionary_war_pension_files_PDFs (4.23 TB)
- https://huggingface.co/datasets/RevolutionCrossroads/si_us_revolutionary_era_collections (7.7 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/RevX_OCR_results/viewer/block_level (19.5 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/RevX_NER_Entity_Occurrences (939 MB)

### Import Libraries

In [ ]:
import numpy as np
import spacy

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances

### Load and Filter Text Dataset

In [ ]:
# Set up dataset stream
ocr_ds = load_dataset("RevolutionCrossroads/RevX_OCR_results",
                      name="block_level",
                      split="train",
                      streaming=True)
ocr_ds.features

In [ ]:
# For filtering data stream
def is_pension(example):
  return example["collection"] == "rev_war_pension"

def is_news(example):
  return example["collection"] == "chronicling_america"

def is_news_block(example):
  return (
    example["collection"] == "chronicling_america" and
    example["block_text"]and
    len(example["block_text"]) > 100
  )

In [ ]:
# Filter stream and get N blocks of text
n_news = 2 ** 14 # 16k
news_ds = ocr_ds.filter(is_news_block)
news_blocks_ds = news_ds.take(n_news)
news_blocks_text = list(news_blocks_ds["block_text"])
len(news_blocks_text)

In [ ]:
# Preprocessing to remove stopwords and punctuation
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

# Custom stopwords
custom_stopwords = set(ENGLISH_STOP_WORDS).union({
  "company",
  "really",
  "also",
  "would",
  "shall",
  "only",
  "ditto",
  "thy",
  "your"
})

def preprocess(text):
  doc = nlp(text.lower())

  return " ".join([
    token.lemma_
    for token in doc
    if not token.is_stop
    and not token.is_punct
    and not token.is_space
    and token not in custom_stopwords
    and len(token) > 2
    and token.is_alpha
  ])

# Pre-process text
docs = [preprocess(doc) for doc in news_blocks_text]

## Query by Embedding

In [ ]:
sentence_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
embeddings = sentence_model.encode(docs, batch_size=32)

In [ ]:
# search docs by term
search_term = "life, liberty and happiness"
# search_term = "monarchy and hereditary rule are fundamentally unjust; people have the natural right to govern themselves."

search_term_emb = sentence_model.encode([search_term])
dists = cosine_distances(search_term_emb, embeddings)[0]
medoid_idxs = dists.argsort()

dist_min, dist_std = np.min(dists), np.std(dists)
thold = dist_min + 1.5 * dist_std

# top-5
for idx in medoid_idxs[:5]:
  print(news_blocks_text[idx])

In [ ]:
# within 1.5 std deviations of min distance
medoid_filtered_idxs = [idx for idx in dists.argsort() if dists[idx] < thold]

print(len(medoid_filtered_idxs))
for idx in medoid_filtered_idxs:
  print(news_blocks_text[idx])

### Another model

In [ ]:
sentence_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", device="cuda")
search_docs = [f"search_document: {d}" for d in docs]
embeddings = sentence_model.encode(search_docs, batch_size=32)

In [ ]:
# search docs by term
search_term = "search_document: life, liberty, freedom, happiness"
# search_term = "search_document: taxation without representation; colonists objected British taxes such as the Stamp Act (1765) and Townshend Acts (1767) because they had no elected representatives in the British Parliament."
# search_term = "search_document: restrictions on westward expansion; colonists seeking land and economic opportunities opposed the proclamation of 1763 which restricted colonial settlement west of the Appalachian Mountains."
# search_term = "search_document: monarchy and hereditary rule are fundamentally unjust; people have the natural right to govern themselves."

search_term_emb = sentence_model.encode([search_term])
dists = cosine_distances(search_term_emb, embeddings)[0]
medoid_idxs = dists.argsort()

dist_min, dist_std = np.min(dists), np.std(dists)
thold = dist_min + 1.5 * dist_std

# top-5
for idx in medoid_idxs[:5]:
  print(news_blocks_text[idx])

In [ ]:
# within 1.5 std deviations of min distance
medoid_filtered_idxs = [idx for idx in dists.argsort() if dists[idx] < thold]

print(len(medoid_filtered_idxs))
for idx in medoid_filtered_idxs:
  print(news_blocks_text[idx])